# Stroke Risk Prediction
## 1. Data Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
print(df.shape)
df.info()

(5110, 12)
<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


In [4]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


Some strange values from above for some of the columns. 
AGE: min age of 0.08, 
BMI: min bmi of 10.3 
Both of these are probably data entry issues.

In [6]:
categorical_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
for col in categorical_cols:
    print(df[col].value_counts(dropna=False))
    print()

gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64

ever_married
Yes    3353
No     1757
Name: count, dtype: int64

work_type
Private          2925
Self-employed     819
children          687
Govt_job          657
Never_worked       22
Name: count, dtype: int64

Residence_type
Urban    2596
Rural    2514
Name: count, dtype: int64

smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64



In [7]:
df.sort_values("bmi",ascending=False)[["age","bmi","stroke"]].head(10)

,age,bmi,stroke
2128,17.0,97.6,0
4209,38.0,92.0,0
928,23.0,78.0,0
544,42.0,71.9,0
1559,53.0,66.8,0
358,52.0,64.8,0
4188,27.0,64.4,0
2764,24.0,63.3,0
3825,52.0,61.6,0
2840,52.0,61.2,0


In [8]:
print(df.groupby(df["bmi"].isna())["stroke"].mean())
df.groupby(df["bmi"].isna())["stroke"].agg(["mean", "count"])


bmi
False    0.042575
True     0.199005
Name: stroke, dtype: float64


,mean,count
bmi,,
False,0.042575,4909
True,0.199005,201


In [9]:
print(df.groupby("smoking_status")["stroke"].mean())
print()
print(df.groupby("smoking_status")["age"].median())

smoking_status
Unknown            0.030440
formerly smoked    0.079096
never smoked       0.047569
smokes             0.053232
Name: stroke, dtype: float64

smoking_status
Unknown            23.0
formerly smoked    57.0
never smoked       47.0
smokes             47.0
Name: age, dtype: float64


### Data quality issues found
- bmi: 201 missing values (4%)
- somking_status: 1544 Unknown, effectively missing
- gender: 1 patient with other
- bmi: some extreme high values (max 97.6) likely errors, 
- age some infants are included in the datasets.

In [10]:
for limit in [30, 40, 50, 60, 70]:
    above = df[df["bmi"] > limit]
    strokes = above["stroke"].sum()
    count = (df["bmi"] > limit).sum()
    print(f"BMI above {limit}: {count} patients, {strokes} had a stroke")

BMI above 30: 1893 patients, 96 had a stroke
BMI above 40: 408 patients, 18 had a stroke
BMI above 50: 79 patients, 1 had a stroke
BMI above 60: 13 patients, 0 had a stroke
BMI above 70: 4 patients, 0 had a stroke


In [11]:
health_cols = ["age", "hypertension", "heart_disease", "avg_glucose_level"]
df.groupby(df["bmi"].isna())[health_cols].mean()

,age,hypertension,heart_disease,avg_glucose_level
bmi,,,,
False,42.865374,0.091872,0.049501,105.305150
True,52.049154,0.233831,0.164179,126.724627


In [12]:
print(df[df["age"] < 10]["smoking_status"].value_counts())
print()
adults = df[df["age"] >= 18 ]
print(adults.groupby("smoking_status")["stroke"].mean())

smoking_status
Unknown    472
Name: count, dtype: int64

smoking_status
Unknown            0.052204
formerly smoked    0.081395
never smoked       0.051370
smokes             0.053846
Name: stroke, dtype: float64


More Data quality discoveries:
extreme BMIs from data entry issues. 
None of the patients with a bmi of above 60 had a stroke so it might make sense to cap the bmi at that level because the people above that bmi, and maybe its a common sense judgment that that feels unlikely.

The missing-bmi patients are sicker across the board, so either it is not entered due to embarrassment. or if that is not thae case, it might be that the patients didn't get weighed and might have been having the stroke at the time, so could be a data leakage issue. 

In [13]:
#check if any of the children actually had a stroke
print(df[df["work_type"] == "children"]["stroke"].value_counts())
#check where age under 16
print(df[df["age"] < 16]["stroke"].value_counts())
#check what age the "children" patients who had a stroke were
print(df[(df["work_type"] == "children") & (df["stroke"] == 1)]["age"].value_counts())
#print all the metrics of those two children who had a stroke
print(df[(df["work_type"] == "children") & (df["stroke"] == 1)])
print(df["work_type"].value_counts())

stroke
0    685
1      2
Name: count, dtype: int64
stroke
0    742
1      2
Name: count, dtype: int64
age
1.32     1
14.00    1
Name: count, dtype: int64
        id  gender    age  hypertension  heart_disease ever_married work_type  \
162  69768  Female   1.32             0              0           No  children   
245  49669  Female  14.00             0              0           No  children   

    Residence_type  avg_glucose_level   bmi smoking_status  stroke  
162          Urban              70.37   NaN        Unknown       1  
245          Rural              57.93  30.9        Unknown       1  
work_type
Private          2925
Self-employed     819
children          687
Govt_job          657
Never_worked       22
Name: count, dtype: int64


## 2. Data Cleaning
I plan on cleaning the data in the following ways after the discoveries made in the exploratory phase.

ID column will be removed. ID is just a unique identifier label, won't have any affect on a patients likelihood of stroke

Gender where value was "Other" will be dropped from the dataset as it is 1 row in 5110 so will not have a major impact on the result of the model. Could pose problems downstream either way as "Other" could be a value that is presented in datasets in the future where the model might struggle due to not having trained on it. the model can't learn a pattern from a single example

BMI for patients with a BMI over 60, we will cap these bmis at 60. There were no strokes recorded for any of the patients with a bmi of over 60, but because we think this might be data entry error we don't want to keep the extremely high values in the dataset. But at least capping them at 60 should minimize the bmi affect on the model for these patients but still allow us to keep the data in the set to use the other metrics in each patients row.

BMI missing: we'll add a "was missing" flag column, the bmi missing is a good flag. We see a 20% stroke rate for bmi missing compared to 4.3%. we Will then fill these in with median bmi, filled with the median from the training data only to avoid data leakage.

smoking_status around 30% of the rows contain "Unknown" as the value for the smoking_status, the median age of this group is 23, so it is likely smoking wasn't recorded for the patients data because they are young. And our dataset has age in it, so it won't attribute "unknown" smoking is the reason for low stroke risk, it will have "age" as a metric to use which is probably the real reason.

Work_type = Children, I think I am going to split out work_type children from the dataset, there are only 2 of 687 where a "children" type experienced a stroke, and one of the lines has no bmi. When it gets cleaned, we would probably give this row a median from the group bmi which would be well inflated for a child of age 1.32. My plan would be to make two models, one for adults only and then one for all of the rows, and maybe see the difference.  Actually I will just use age and remove under 18s from the dataset. there are still only 2 from the 856 under 18 patients. and we are only taking out two of 249 rows with the target metric.

Children are an easy negative for the model, because only 2 of 687 had stroked. Taking out the children might lower the accuracy score from the model but will be training better on the important features. and not skewed by having many young patients who are easy "no"s to stroke risk.

In [19]:
# Check for dupes
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate ids:", df["id"].duplicated().sum())

Duplicate rows: 0
Duplicate ids: 0


In [ ]:
#remove the "Other" gender category based on decision described above. to avoid issues
#with stratification because when we do that we cannot have only one row of that type
#because stratification have the same proportion of each category in the train and test sets. 
# So if we have only one row of a category, it will not be present in both the train and test sets.
df_clean = df.copy()
df_clean = df_clean[df_clean["gender"] != "Other"]
df_clean = df_clean[df_clean["age"] >= 18]## ####~~efewwedwqdwqfewkjncekvjnewjkfwefwefew

In [21]:
#add missing bmi flag
df_clean["bmi_missing"] = df_clean["bmi"].isna().astype(int)
#Check that worked
print(df_clean[["bmi_missing","bmi"]])

      bmi_missing   bmi
0               0  36.6
1               1   NaN
2               0  32.5
3               0  34.4
4               0  24.0
...           ...   ...
5105            1   NaN
5106            0  40.0
5107            0  30.6
5108            0  25.6
5109            0  26.2

[4253 rows x 2 columns]


In [22]:
#cap bmi at 60 and remove id
df_clean["bmi"] = df_clean["bmi"].clip(upper=60)
df_clean = df_clean.drop(columns=["id"])

In [23]:
#check all the cleaning worked
print(df_clean.shape)
# print(df_clean.head(10))
# print(df.shape)
print(df_clean["gender"].value_counts())
print("Max BMI:", df_clean["bmi"].max())
print("Missing BMI:", df_clean["bmi"].isna().sum())
print("Flag totals:", df_clean["bmi_missing"].sum())

(4253, 12)
gender
Female    2576
Male      1677
Name: count, dtype: int64
Max BMI: 60.0
Missing BMI: 181
Flag totals: 181
